## 0. Prerequisites

**Software**
- Python 3.10+
- A running **Neo4j 5+** instance reachable at `bolt://127.0.0.1:7687`
  (Neo4j Desktop, or Docker: `docker run -p 7687:7687 -p 7474:7474 -e NEO4J_AUTH=neo4j/yourpassword neo4j:5`)
- An OpenAI API key

**Python packages** (run once):

In [6]:
%pip install langchain-core langchain-neo4j langchain-openai langchain-experimental langchain-text-splitters python-dotenv neo4j pydantic

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration — values come from your `.env` file

Nothing secret is written in this notebook. We call `load_dotenv()`, which reads a `.env`
file sitting next to the notebook, and pull every value from the environment.

Your `.env` should contain:

```
OPENAI_API_KEY=your-key-here
NEO4J_URI=bolt://127.0.0.1:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your-neo4j-password
NEO4J_DATABASE=trail_demo
```

The cell below leaves the values **blank by design** — it only reads them. If a required
value is missing, it stops immediately with a clear message instead of failing deep inside
a library call later.

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current folder

# --- Secrets / connection settings (filled from .env, never hardcoded) -------
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "_9aQWZwq2Kj4A").strip()
NEO4J_URI      = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "enter ur pwd")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "batch3demo")

# Fail loudly if anything required is missing.
_missing = [k for k, v in {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_USERNAME": NEO4J_USERNAME,
    "NEO4J_PASSWORD": NEO4J_PASSWORD,
    "NEO4J_DATABASE": NEO4J_DATABASE,
}.items() if not v]

if _missing:
    raise SystemExit(f"Missing in .env: {', '.join(_missing)}")

print("Config loaded from .env. Database:", NEO4J_DATABASE)

Config loaded from .env. Database: batch3demo


## 2. Imports

We use **LangChain** as the glue: `langchain-openai` for the embedding + chat models,
`langchain-neo4j` for the graph store and vector store, and
`langchain-experimental`'s `LLMGraphTransformer` to turn free text into graph nodes/edges.

In [15]:
import re
import textwrap
from operator import itemgetter
from typing import List

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import TokenTextSplitter
from pydantic import BaseModel, Field

## 3. Make sure the target database exists

Neo4j 5 supports multiple named databases. This helper creates the one named in
`NEO4J_DATABASE` if it isn't there yet. It runs against the special `system` database,
which is where database-management commands live.

> **Why the regex check?** We interpolate the database name into a Cypher string, so we
> validate it first to avoid injection via a malformed name.

In [16]:
def ensure_database_exists() -> None:
    """Create the named database if missing (Neo4j 5+, requires system permission)."""
    if not re.match(r"^[a-zA-Z][a-zA-Z0-9_-]*$", NEO4J_DATABASE):
        raise ValueError(f"Unsafe NEO4J_DATABASE name: {NEO4J_DATABASE!r}")
    try:
        from neo4j import GraphDatabase
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
        with driver.session(database="system") as session:
            session.run(f"CREATE DATABASE `{NEO4J_DATABASE}` IF NOT EXISTS")
        driver.close()
        print(f"Ensured database exists: {NEO4J_DATABASE}")
    except Exception as exc:
        print(f"Note: could not auto-create database ({exc!r}). "
              f"Create `{NEO4J_DATABASE}` manually if the connection fails.")


ensure_database_exists()

Ensured database exists: batch3demo


## 4. Connect the models and the graph

Three objects:
- **`embeddings`** — turns text into vectors (`text-embedding-3-small`) for similarity search.
- **`chat`** — the LLM (`gpt-4o-mini`, `temperature=0` for deterministic, repeatable answers).
- **`kg`** — the live connection to the Neo4j database. `refresh_schema()` reads the current
  node labels / relationship types so LangChain knows the shape of the graph.

In [17]:
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY, model="text-embedding-3-small")
chat = ChatOpenAI(api_key=OPENAI_API_KEY, temperature=0, model="gpt-4o-mini")

kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)
kg.refresh_schema()
print("Connection established and schema refreshed.")

Connection established and schema refreshed.


## 5. The corpus — three business-policy documents

These are deliberately **dense, rule-heavy** documents. Each one packs in named entities
(companies, batches, contracts), numbers (thresholds, dates), and **conditional rules**
("if excursion ≤ 25 minutes AND written exception AND risk form → pre-release"). That kind
of content is exactly where GraphRAG earns its keep: the answer often hinges on a chain of
related facts, not a single paragraph.

The three domains:
1. **MedFridge / NordicCare** — pharma cold-chain release rules.
2. **HelioStack / ACME** — a SaaS enterprise agreement + revenue recognition.
3. **FobCo / RhineChem** — a trade-finance letter of credit with an Incoterms discrepancy.

In [18]:
raw_documents = [
    Document(
        page_content=(
            "POLICY PACK: MedFridge Ltd - Cold-chain oncology injectables (EU GDP). "
            "SKUs: MF-Insulin-Pro (2-8 deg C), MF-MAB-Cool (-25 to -15 deg C frozen). "
            "Batch B-4421 (MF-Insulin-Pro) was released from Plant Dresden on 2025-11-02 "
            "with a stability budget of 48 aggregate excursion-minutes above 8 deg C before "
            "lot disposition must be escalated to QP (Qualified Person). Courier Apex "
            "Cold carried the pallet to distributor NordicCare. NordicCare's receiving "
            "SOP states: if logger shows any single contiguous excursion >20 minutes "
            "above 8 deg C, the pallet is QUARANTINED and cannot be merged with sellable "
            "inventory until QA completes investigation. However, NordicCare may "
            "PRE-RELEASE to hospital back-order if (a) the excursion is at most 25 minutes, "
            "(b) MedFridge Medical Affairs emails written exception referencing batch "
            "stability report SR-B4421-REV-C, and (c) the hospital accepts residual "
            "risk form RF-77. Batch B-4408 is unrelated (different stability report). "
            "Regulatory: EMA variation VAR-1189 caps combined truck + warehouse hand-off "
            "delay at 36 hours for MF-Insulin-Pro; Apex logged 31 hours end-to-end for "
            "B-4421. If quarantine is triggered, NordicCare must notify EudraVigilance "
            "only when product has already left quarantine to a patient - not while "
            "held on-site. Finance rule: revenue for B-4421 cannot be recognized at "
            "MedFridge until NordicCare posts 'Available for sale' in ERP ledger code "
            "NCF-AS-01."
        ),
        metadata={
            "title": "MedFridge cold-chain and NordicCare release rules",
            "source": "business:medfridge-policy-pack",
            "domain": "pharma_logistics",
        },
    ),
    Document(
        page_content=(
            "COMMERCIAL MEMO: HelioStack SaaS - Enterprise Agreement EA-2024-771 with "
            "ACME Corp (manufacturing vertical). Contract term 2024-07-01 to 2027-06-30. "
            "Committed ARR: $1.2M for 2,400 named seats of module 'HelioMES'; unit list "
            "price $600/seat/year before tiered discount. Clause 14(b): at each renewal "
            "anniversary, customer may downgrade up to 20% of committed seats without "
            "termination fee; downgraded seats convert to month-to-month list price for "
            "remainder of term unless re-committed. Clause 14(c): if downgrade exceeds "
            "20%, HelioStack may terminate for convenience with 90-day notice and forfeit "
            "only unbilled future periods (no clawback of cash already collected). "
            "Revenue policy (ASC 606 memo FIN-HS-09): for multi-year prepay deals, "
            "HelioStack recognizes revenue straight-line over the service period; "
            "if seats are removed mid-period, deferred revenue is reduced prospectively "
            "from the downgrade effective date - never retroactively restating prior "
            "quarters. Customer success owns 'adoption score' gates: if adoption score "
            "<40 at day-180, auto-renew uplift of 7% is waived. ACME's plant in "
            "Gdansk shares one tenant with subsidiary ACME Baltics; Baltics seats are "
            "counted inside the same 2,400 pool (not additive). Competitor OrionMES is "
            "excluded from data residency routing per Annex D."
        ),
        metadata={
            "title": "HelioStack ACME enterprise agreement and revenue rules",
            "source": "business:heliostack-ea-771",
            "domain": "b2b_saas_contracts",
        },
    ),
    Document(
        page_content=(
            "TRADE OPS BRIEF: FobCo (Mumbai exporter) sold specialty dyes to RhineChem "
            "AG under contract CTR-RH-303. Payment: irrevocable letter of credit LCI-9001 "
            "issued by Deutsche Handelsbank (confirming bank added). Required documents "
            "under UCP 600 field 46A: commercial invoice, packing list, certificate of "
            "origin, full set onboard bill of lading showing shipment from Nhava Sheva "
            "to Hamburg, and phytosanitary certificate. Contract Incoterms 2020: CIF "
            "Hamburg - seller arranges carriage and insurance to named port; risk "
            "transfers when goods pass ship's rail at origin port per Incoterms rules. "
            "The B/L received by the negotiating bank shows 'FOB Nhava Sheva' and "
            "freight 'collect'. Discrepancy playbook: if Incoterms on B/L contradict "
            "contract, RhineChem's treasury may ACCEPT waiver if RhineChem signs "
            "discrepancy indemnity DI-IND-01 before presentation; otherwise documents "
            "must be re-issued. Force majeure addendum FM-22: port worker strikes at "
            "Hamburg do NOT excuse FobCo from presenting conforming documents - they only "
            "extend delivery tolerance by 10 calendar days for physical arrival, not for "
            "documentary compliance. Sanctions screen: RhineChem is Tier-1 cleared; "
            "transhipment via sanctioned corridor SC-List-B is forbidden even if cheaper."
        ),
        metadata={
            "title": "FobCo RhineChem LC and Incoterms discrepancy handling",
            "source": "business:fobco-trade-ctr-rh-303",
            "domain": "trade_finance",
        },
    ),
]
print(f"Loaded {len(raw_documents)} source document(s).")

Loaded 3 source document(s).


## 6. Chunking

LLMs and embedding models work better on bounded pieces of text. `TokenTextSplitter`
splits on **token** boundaries (not characters), so each chunk is a predictable size for
the model. `chunk_overlap=24` repeats a few tokens between neighbours so a rule that
straddles a boundary isn't cut in half.

In [19]:
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents)

print(f"Loaded {len(documents)} chunk(s) from {len(raw_documents)} document(s).")
print("Sample chunk metadata:", [d.metadata.get("title") for d in documents[:3]])

Loaded 3 chunk(s) from 3 document(s).
Sample chunk metadata: ['MedFridge cold-chain and NordicCare release rules', 'HelioStack ACME enterprise agreement and revenue rules', 'FobCo RhineChem LC and Incoterms discrepancy handling']


## 7. Build the knowledge graph

This is the GraphRAG-specific step. `LLMGraphTransformer` asks the LLM to read each chunk
and emit **(entity)-[relationship]->(entity)** triples — e.g.
`(NordicCare)-[QUARANTINED]->(Batch B-4421)`. We then write those into Neo4j.

- `include_source=True` keeps a link from each extracted node back to the source `Document`
  (via a `MENTIONS` relationship), so you can trace any fact to its origin.
- `baseEntityLabel=True` adds a shared `__Entity__` label to every node, which makes the
  fulltext index in the next section simple to define.

> **Cost note:** this calls the LLM once per chunk, so it's the slowest/most expensive cell.
> You only need to run it when the corpus changes.

In [20]:
llm_transformer = LLMGraphTransformer(llm=chat)
graph_documents = llm_transformer.convert_to_graph_documents(documents)

# Inspect what was extracted (nodes + relationships per chunk).
for gd in graph_documents:
    print("Nodes:", [n.id for n in gd.nodes])
    print("Rels :", [(r.source.id, r.type, r.target.id) for r in gd.relationships])
    print("-" * 60)

kg.add_graph_documents(
    graph_documents,
    include_source=True,
    baseEntityLabel=True,
)
print("Graph written to Neo4j.")

Nodes: ['Medfridge Ltd', 'Cold-Chain Oncology Injectables', 'Mf-Insulin-Pro', 'Mf-Mab-Cool', 'Batch B-4421', 'Plant Dresden', 'Apex Cold', 'Nordiccare', 'Eudravigilance', 'Batch B-4408', 'Ema Variation Var-1189', 'Sr-B4421-Rev-C', 'Rf-77']
Rels : [('Batch B-4421', 'IS_PART_OF', 'Mf-Insulin-Pro'), ('Batch B-4421', 'RELEASED_FROM', 'Plant Dresden'), ('Batch B-4421', 'CARRIED_BY', 'Apex Cold'), ('Apex Cold', 'DELIVERED_TO', 'Nordiccare'), ('Nordiccare', 'NOTIFY_IF_QUARANTINE_TRIGGERED', 'Eudravigilance'), ('Batch B-4421', 'REFERENCED_BY', 'Sr-B4421-Rev-C'), ('Batch B-4421', 'ACCEPTS_RESIDUAL_RISK', 'Rf-77'), ('Ema Variation Var-1189', 'CAPS_DELAY', 'Mf-Insulin-Pro'), ('Batch B-4421', 'REVENUE_RECOGNIZED_AT', 'Nordiccare'), ('Nordiccare', 'UNRELATED_TO', 'Batch B-4408')]
------------------------------------------------------------
Nodes: ['Heliostack Saas', 'Acme Corp', 'Gdansk', 'Acme Baltics', 'Orionmes', 'Clause 14(B)', 'Clause 14(C)', 'Asc 606']
Rels : [('Acme Corp', 'ENTERED_INTO_AGRE

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.addLabels is deprecated. It is replaced by Cypher's dynamic labels; `SET n:$(labels)`..", position=<SummaryInputPosition line=1, column=257, offset=256>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 256, 'line': 1, 'column': 257}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MERGE (d:Document {id:$document.metadata.id}) SET d.text = $document.page_content SET d += $document.metadata WITH d UNWIND $data AS row MERGE (source:`__Entity__` {id: row.id}) SET source += row.properties MERGE (d)-[:MENTIONS]->(source) WITH source, row CALL apoc.create.addLabels( source, [row.type] 

Graph written to Neo4j.


## 8. Build the vector index

The *other* retrieval path. `Neo4jVector.from_documents` embeds every chunk and stores the
vectors **inside the same Neo4j database**, on nodes labelled `Document`. Later we can run a
similarity search to fetch the chunks most semantically related to a question.

`pre_delete_collection=True` clears any previous version of this index so re-runs start clean.

In [21]:
vector_index = Neo4jVector.from_documents(
    documents=documents,
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    index_name="graphrag_classnote_vector",
    node_label="Document",
    embedding_node_property="embedding",
    text_node_property="text",
    pre_delete_collection=True,
)
print("Vector index built.")

Vector index built.


## 9. Question → entities

To traverse the graph for a question, we first need to know **which entities the question is
about**. We ask the LLM to extract them, using `with_structured_output` so the result is a
typed `Entities` object (a list of names) rather than free text we'd have to parse.

In [22]:
class Entities(BaseModel):
    """Identifying information about entities."""
    names: List[str] = Field(
        ...,  # required field: the model must always return a list
        description="Person, organization, product, or key technical concept "
                    "names that appear in the text (for graph-aware retrieval).",
    )


prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You extract person, organization, product, and notable concept entities "
     "from the user's question for knowledge-graph lookup."),
    ("human", "Extract entities from: {question}"),
])

entity_chain = prompt | chat.with_structured_output(Entities)

# Quick check:
entity_chain.invoke({"question": "What rule applies to NordicCare and Batch B-4421?"})

Entities(names=['NordicCare', 'Batch B-4421'])

## 10. A fulltext index + a fuzzy-query helper

The entity names the LLM extracts won't always match the graph exactly (casing, typos,
slight wording differences). So we:

1. Create a **fulltext index** named `entity` over every `__Entity__` node's `id`.
2. Convert each extracted name into a **Lucene fuzzy query** — `MedFridge~2` means "match
   `MedFridge` allowing up to 2 character edits". Multiple words are joined with `AND`.

`remove_lucene_chars` strips characters like `+ - !` that would otherwise break Lucene syntax.

In [23]:
kg.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]"
)


def generate_full_text_query(input: str) -> str:
    """Convert free text into a Lucene fuzzy query.

    Example: "MedFridge NordicCare" -> "MedFridge~2 AND NordicCare~2"
    - ~2  = fuzzy match, edit distance 2 (tolerates typos / spelling variants)
    - AND = every word must match
    """
    words = [el for el in remove_lucene_chars(input).split() if el]
    if not words:
        return ""
    query = ""
    for word in words[:-1]:
        query += f" {word}~2 AND"
    query += f" {words[-1]}~2"
    return query.strip()


print(generate_full_text_query("MedFridge NordicCare"))

MedFridge~2 AND NordicCare~2


## 11. Structured retriever (graph traversal)

This is the heart of the "graph" half. For a question it:

1. Extracts entity names (cell 9).
2. Finds each one in the fulltext index (cell 10).
3. **Traverses one hop** out from and into each matched node, collecting relationship
   triples like `NordicCare - QUARANTINED -> Batch B-4421`.

The `!MENTIONS` filter excludes the bookkeeping link back to source documents — we only want
the *meaningful* domain relationships. The `UNION ALL` gathers **both** outgoing and incoming
edges so we see the full neighbourhood.

In [24]:
def structured_retriever(question: str) -> str:
    """Return newline-separated relationship triples relevant to the question."""
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        print(f"  Getting entity: {entity}")
        response = kg.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node, score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el["output"] for el in response])
    return result

## 12. Hybrid retriever — graph **+** vectors

Now combine both paths into one context string:

- **Structured data:** the relationship triples from `structured_retriever`.
- **Unstructured data:** the top semantically-similar chunks from the vector index.

Giving the LLM both means it can reason over explicit relationships *and* fall back on the
raw policy wording for the details.

In [25]:
def retriever(question: str) -> str:
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [
        el.page_content for el in vector_index.similarity_search(question)
    ]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{chr(10).join(f"--- doc chunk ---{chr(10)}{t}" for t in unstructured_data)}
"""
    print(f"\n--- Retrieved context preview (first ~1800 chars) ---\n{final_data[:1800]}...")
    return final_data

## 13. The QA chain

We assemble a LangChain pipeline with the `|` operator:

```
{context, question}  ->  prompt  ->  chat  ->  plain string
```

- `RunnableParallel` builds the input dict: `context` runs our `retriever`, `question` is
  passed through unchanged.
- The prompt **constrains the model** to answer only from the retrieved context and to say
  what's missing rather than invent — important for policy/compliance questions.

In [26]:
template = """Answer using ONLY the context below. Apply stated policies, numbers, and conditional rules explicitly.
If the context is insufficient, say what is missing - do not invent facts.

Context:
{context}

Question: {question}

Answer (clear and concise):"""

qa_prompt = ChatPromptTemplate.from_template(template)

chain = (
    RunnableParallel(
        {
            "context": RunnableLambda(itemgetter("question")) | RunnableLambda(retriever),
            "question": RunnableLambda(itemgetter("question")),
        }
    )
    | qa_prompt
    | chat
    | StrOutputParser()
)
print("Chain ready.")

Chain ready.


## 14. Demo questions

Each question is designed to require **combining facts** — a single similarity search usually
isn't enough. The last one is a deliberate cross-check against the same corpus.

In [27]:
DEMO_QUESTIONS = [
    (
        "MedFridge / NordicCare",
        "Batch B-4421 (MF-Insulin-Pro) had a single contiguous temperature excursion "
        "of 22 minutes above 8 deg C before NordicCare received it. Per the policy pack, "
        "must the pallet be quarantined, or can it be pre-released to a hospital "
        "back-order - and what three conditions must be met for pre-release?",
    ),
    (
        "HelioStack / ACME",
        "Under EA-2024-771 and revenue memo FIN-HS-09, if ACME downgrades exactly 20% "
        "of committed HelioMES seats effective mid-quarter (not exceeding 20%), how is "
        "recognized revenue adjusted - retrospectively or prospectively - and what "
        "happens to cash already collected?",
    ),
    (
        "FobCo / RhineChem LC",
        "Bill of lading shows FOB Nhava Sheva and freight collect, but contract CTR-RH-303 "
        "is CIF Hamburg. Can RhineChem's treasury still accept documents to pay under "
        "LCI-9001 without re-issued B/L - if yes, what must happen first?",
    ),
    (
        "Cross-check (same corpus)",
        "Does MedFridge recognize revenue for batch B-4421 when NordicCare only "
        "quarantines the pallet (no 'Available for sale' in NCF-AS-01)? Answer yes/no "
        "and cite the rule.",
    ),
]


def print_qa_block(title: str, question: str, answer: str) -> None:
    width = 78
    print("\n" + "=" * width)
    print(f" TOPIC: {title}")
    print("=" * width)
    print("QUESTION:")
    print(textwrap.fill(question, width=width))
    print("-" * width)
    print("ANSWER:")
    print(textwrap.fill(answer.strip(), width=width))
    print("=" * width)

## 15. Smoke test — just the graph traversal

Before running the full chain, sanity-check that entity extraction + graph traversal return
*something*. If this prints no lines, the issue is upstream (graph not built, or names not
matching) — useful to isolate before involving the LLM answer step.

In [28]:
_title0, q0 = DEMO_QUESTIONS[0]
print(structured_retriever(q0)[:1200] or "(no graph lines for extracted entities)")

  Getting entity: Batch B-4421


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            

  Getting entity: MF-Insulin-Pro
  Getting entity: NordicCare
Batch B-4421 - IS_PART_OF -> Mf-Insulin-Pro
Batch B-4421 - RELEASED_FROM -> Plant Dresden
Batch B-4421 - CARRIED_BY -> Apex Cold
Batch B-4421 - REFERENCED_BY -> Sr-B4421-Rev-C
Batch B-4421 - ACCEPTS_RESIDUAL_RISK -> Rf-77
Batch B-4421 - REVENUE_RECOGNIZED_AT -> Nordiccare
Nordiccare - UNRELATED_TO -> Batch B-4408Batch B-4421 - IS_PART_OF -> Mf-Insulin-Pro
Ema Variation Var-1189 - CAPS_DELAY -> Mf-Insulin-ProNordiccare - NOTIFY_IF_QUARANTINE_TRIGGERED -> Eudravigilance
Nordiccare - UNRELATED_TO -> Batch B-4408
Apex Cold - DELIVERED_TO -> Nordiccare
Batch B-4421 - REVENUE_RECOGNIZED_AT -> Nordiccare


## 16. Run the full GraphRAG chain

For each question this triggers the whole pipeline:
**entity extraction → graph traversal + vector search → grounded LLM answer.**

In [29]:
answers = []
for idx, (topic_title, question) in enumerate(DEMO_QUESTIONS, start=1):
    print(f"\n>>> Running chain for question {idx}/{len(DEMO_QUESTIONS)}...")
    ans = chain.invoke({"question": question})
    answers.append(ans)
    print_qa_block(topic_title, question, ans)

# Sanity check: no empty answers.
for i, ans in enumerate(answers, start=1):
    assert ans.strip(), f"Empty answer for question {i}"
print("\nSelf-check passed: all answers non-empty.")


>>> Running chain for question 1/4...
Search query: Batch B-4421 (MF-Insulin-Pro) had a single contiguous temperature excursion of 22 minutes above 8 deg C before NordicCare received it. Per the policy pack, must the pallet be quarantined, or can it be pre-released to a hospital back-order - and what three conditions must be met for pre-release?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            

  Getting entity: Batch B-4421
  Getting entity: MF-Insulin-Pro
  Getting entity: NordicCare


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN node.`text` AS text, score, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'



--- Retrieved context preview (first ~1800 chars) ---
Structured data:
Batch B-4421 - IS_PART_OF -> Mf-Insulin-Pro
Batch B-4421 - RELEASED_FROM -> Plant Dresden
Batch B-4421 - CARRIED_BY -> Apex Cold
Batch B-4421 - REFERENCED_BY -> Sr-B4421-Rev-C
Batch B-4421 - ACCEPTS_RESIDUAL_RISK -> Rf-77
Batch B-4421 - REVENUE_RECOGNIZED_AT -> Nordiccare
Nordiccare - UNRELATED_TO -> Batch B-4408Batch B-4421 - IS_PART_OF -> Mf-Insulin-Pro
Ema Variation Var-1189 - CAPS_DELAY -> Mf-Insulin-ProNordiccare - NOTIFY_IF_QUARANTINE_TRIGGERED -> Eudravigilance
Nordiccare - UNRELATED_TO -> Batch B-4408
Apex Cold - DELIVERED_TO -> Nordiccare
Batch B-4421 - REVENUE_RECOGNIZED_AT -> Nordiccare
Unstructured data:
--- doc chunk ---
POLICY PACK: MedFridge Ltd - Cold-chain oncology injectables (EU GDP). SKUs: MF-Insulin-Pro (2-8 deg C), MF-MAB-Cool (-25 to -15 deg C frozen). Batch B-4421 (MF-Insulin-Pro) was released from Plant Dresden on 2025-11-02 with a stability budget of 48 aggregate excursion-minutes above 8 

...


--- Retrieved context preview (first ~1800 chars) ---
Structured data:
Contract Ctr-Rh-303 - INCLUDES -> Fm-22
Contract Ctr-Rh-303 - CONTRACT_WITH -> Rhinechem Ag
Contract Ctr-Rh-303 - GOVERNED_BY -> Ucp 600
Fobco - SOLD -> Contract Ctr-Rh-303Rhinechem Ag - CLEARED -> Sc-List-B
Contract Ctr-Rh-303 - CONTRACT_WITH -> Rhinechem AgLci-9001 - ISSUED_BY -> Deutsche Handelsbank
Contract Ctr-Rh-303 - GOVERNED_BY -> Ucp 600
Unstructured data:
--- doc chunk ---
TRADE OPS BRIEF: FobCo (Mumbai exporter) sold specialty dyes to RhineChem AG under contract CTR-RH-303. Payment: irrevocable letter of credit LCI-9001 issued by Deutsche Handelsbank (confirming bank added). Required documents under UCP 600 field 46A: commercial invoice, packing list, certificate of origin, full set onboard bill of lading showing shipment from Nhava Sheva to Hamburg, and phytosanitary certificate. Contract Incoterms 2020: CIF Hamburg - seller arranges carriage and insurance to named port; risk transfers when goods pass s

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=105>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 105, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})\n            YIELD node, score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n            

  Getting entity: MedFridge
  Getting entity: NordicCare
  Getting entity: NCF-AS-01
  Getting entity: batch B-4421


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN node.`text` AS text, score, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'



--- Retrieved context preview (first ~1800 chars) ---
Structured data:
Nordiccare - NOTIFY_IF_QUARANTINE_TRIGGERED -> Eudravigilance
Nordiccare - UNRELATED_TO -> Batch B-4408
Apex Cold - DELIVERED_TO -> Nordiccare
Batch B-4421 - REVENUE_RECOGNIZED_AT -> NordiccareBatch B-4421 - IS_PART_OF -> Mf-Insulin-Pro
Ema Variation Var-1189 - CAPS_DELAY -> Mf-Insulin-ProBatch B-4421 - IS_PART_OF -> Mf-Insulin-Pro
Batch B-4421 - RELEASED_FROM -> Plant Dresden
Batch B-4421 - CARRIED_BY -> Apex Cold
Batch B-4421 - REFERENCED_BY -> Sr-B4421-Rev-C
Batch B-4421 - ACCEPTS_RESIDUAL_RISK -> Rf-77
Batch B-4421 - REVENUE_RECOGNIZED_AT -> Nordiccare
Nordiccare - UNRELATED_TO -> Batch B-4408
Unstructured data:
--- doc chunk ---
POLICY PACK: MedFridge Ltd - Cold-chain oncology injectables (EU GDP). SKUs: MF-Insulin-Pro (2-8 deg C), MF-MAB-Cool (-25 to -15 deg C frozen). Batch B-4421 (MF-Insulin-Pro) was released from Plant Dresden on 2025-11-02 with a stability budget of 48 aggregate excursion-minutes above 8 

## Recap & things to try

**What you built:** a hybrid retriever that fuses a Neo4j knowledge graph (explicit
relationships) with a vector index (semantic similarity), feeding both into an LLM that is
constrained to answer only from retrieved context.

**Exercises:**
- Add a fourth document and a question that requires linking it to an existing entity.
- Print the retrieved `structured_data` vs `unstructured_data` separately and see which one
  actually carries the answer for each demo question.
- Set `temperature` above 0 and observe how answers drift — then argue why 0 is right here.
- Swap `gpt-4o-mini` for a larger model and compare the extracted graph in cell 7.